# Visualize a book's chunks in Chroma

Reads whatever's already embedded (via `app.py`) and plots it with t-SNE. Doesn't call any AI or touch Gradio - just reads the vector store.

Set `EMB_MODEL` to whichever embedding model you embedded the book with, run the next cell to see which books are available in that collection, then set `BOOK_NAME` to one of them.

In [7]:
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from dotenv import load_dotenv

from auto_regressive_llm import get_collection

load_dotenv(override=True)

ERROR: Could not find a version that satisfies the requirement sklearn.manifold (from versions: none)
ERROR: No matching distribution found for sklearn.manifold


True

In [8]:
EMB_MODEL = "gemini/text-embedding-004"

collection = get_collection(EMB_MODEL)
all_metadatas = collection.get(include=["metadatas"])["metadatas"]
available_books = sorted(set(metadata["name"] for metadata in all_metadatas))
available_books

[]

In [9]:
BOOK_NAME = available_books[0]

result = collection.get(where={"name": BOOK_NAME}, include=["embeddings", "documents", "metadatas"])
vectors = np.array(result["embeddings"])
documents = result["documents"]

print(f"'{BOOK_NAME}': {len(documents)} chunks")

IndexError: list index out of range

Points are colored by chunk order (0 = start of the book) so you can see whether adjacent chunks land near each other in embedding space.

In [ ]:
perplexity = min(30, max(5, len(vectors) - 1))
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
reduced_vectors = tsne.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode="markers",
    marker=dict(size=5, color=list(range(len(documents))), colorscale="Viridis", opacity=0.8, colorbar=dict(title="Chunk order")),
    text=[f"Chunk {i}<br>{d[:150]}..." for i, d in enumerate(documents)],
    hoverinfo="text",
)])

fig.update_layout(
    title=f"2D Chunk Embeddings - {BOOK_NAME}",
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40),
)

fig.show()

In [ ]:
tsne_3d = TSNE(n_components=3, random_state=42, perplexity=perplexity)
reduced_vectors_3d = tsne_3d.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors_3d[:, 0],
    y=reduced_vectors_3d[:, 1],
    z=reduced_vectors_3d[:, 2],
    mode="markers",
    marker=dict(size=5, color=list(range(len(documents))), colorscale="Viridis", opacity=0.8, colorbar=dict(title="Chunk order")),
    text=[f"Chunk {i}<br>{d[:150]}..." for i, d in enumerate(documents)],
    hoverinfo="text",
)])

fig.update_layout(
    title=f"3D Chunk Embeddings - {BOOK_NAME}",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40),
)

fig.show()